⚠️ This notebook is for experimentation and analysis only.
Production logic is implemented under `src/` and `pipeline/`.


⚠️ Note on Time-Series Grain

During experimentation, item-level modeling was performed across all stores.
In the production pipeline, this was corrected to use (store_id, item_id) as the
true time-series grain to prevent cross-store data leakage.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing


In [2]:
INPUT_PATH = "../data/processed/train_fe.csv"
OUTPUT_DIR = Path("../data/processed/baselines")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df = df.sort_values(["item_id", "date"]).reset_index(drop=True)


In [3]:
HORIZON = 28

train_df = (
    df.groupby("item_id")
      .apply(lambda x: x.iloc[:-HORIZON])
      .reset_index(drop=True)
)

valid_df = (
    df.groupby("item_id")
      .apply(lambda x: x.iloc[-HORIZON:])
      .reset_index(drop=True)
)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_37496\2974896321.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:-HORIZON])
C:\Users\ASUS\AppData\Local\Temp\ipykernel_37496\2974896321.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[-HORIZON:])


In [4]:
SAMPLE_ITEMS = train_df["item_id"].unique()[:10]


In [5]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }


In [6]:
arima_preds = []

for item_id in SAMPLE_ITEMS:
    train_series = train_df[train_df["item_id"] == item_id]["sales"]
    valid_series = valid_df[valid_df["item_id"] == item_id]["sales"]

    try:
        model = ARIMA(train_series, order=(1,1,1))
        fitted = model.fit()
        forecast = fitted.forecast(HORIZON)

        arima_preds.append(pd.DataFrame({
            "date": valid_df[valid_df["item_id"] == item_id]["date"].values,
            "store_id": valid_df[valid_df["item_id"] == item_id]["store_id"].values,
            "item_id": item_id,
            "actual": valid_series.values,
            "pred_arima": forecast.values
        }))
    except:
        continue


c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will

In [7]:
sarima_preds = []

for item_id in SAMPLE_ITEMS:
    train_series = train_df[train_df["item_id"] == item_id]["sales"]
    valid_series = valid_df[valid_df["item_id"] == item_id]["sales"]

    try:
        model = SARIMAX(
            train_series,
            order=(1,1,1),
            seasonal_order=(1,1,1,7),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted = model.fit(disp=False)
        forecast = fitted.forecast(HORIZON)

        sarima_preds.append(pd.DataFrame({
            "date": valid_df[valid_df["item_id"] == item_id]["date"].values,
            "store_id": valid_df[valid_df["item_id"] == item_id]["store_id"].values,
            "item_id": item_id,
            "actual": valid_series.values,
            "pred_sarima": forecast.values
        }))
    except:
        continue


c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a support

In [8]:
ets_preds = []

for item_id in SAMPLE_ITEMS:
    train_series = train_df[train_df["item_id"] == item_id]["sales"]
    valid_series = valid_df[valid_df["item_id"] == item_id]["sales"]

    try:
        model = ExponentialSmoothing(
            train_series,
            trend="add",
            seasonal="add",
            seasonal_periods=7
        )

        fitted = model.fit()
        forecast = fitted.forecast(HORIZON)

        ets_preds.append(pd.DataFrame({
            "date": valid_df[valid_df["item_id"] == item_id]["date"].values,
            "store_id": valid_df[valid_df["item_id"] == item_id]["store_id"].values,
            "item_id": item_id,
            "actual": valid_series.values,
            "pred_ets": forecast.values
        }))
    except:
        continue


c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\ASUS\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use on

In [9]:
predictions_df = (
    pd.concat(arima_preds + sarima_preds + ets_preds, axis=0)
    .reset_index(drop=True)
)

predictions_df.to_csv(
    OUTPUT_DIR / "classical_baseline_predictions.csv",
    index=False
)


In [11]:
metrics = {}

# ARIMA
arima_mask = predictions_df["pred_arima"].notna()
metrics["ARIMA"] = evaluate(
    predictions_df.loc[arima_mask, "actual"],
    predictions_df.loc[arima_mask, "pred_arima"]
)

# SARIMA
sarima_mask = predictions_df["pred_sarima"].notna()
metrics["SARIMA"] = evaluate(
    predictions_df.loc[sarima_mask, "actual"],
    predictions_df.loc[sarima_mask, "pred_sarima"]
)

# ETS
ets_mask = predictions_df["pred_ets"].notna()
metrics["ETS"] = evaluate(
    predictions_df.loc[ets_mask, "actual"],
    predictions_df.loc[ets_mask, "pred_ets"]
)



In [12]:
metrics_df = (
    pd.DataFrame(metrics)
    .T
    .reset_index()
    .rename(columns={"index": "model"})
)

metrics_df.to_csv(
    OUTPUT_DIR / "classical_baseline_metrics.csv",
    index=False
)


## Classical Baseline Models – Summary

Classical time-series models (ARIMA, SARIMA, and Holt-Winters ETS) were evaluated
on a representative subset of item-level series using a 28-day forecasting horizon.

Due to computational constraints and lack of scalability, these models were
applied to sampled items only. While they capture trend and seasonality,
their performance and scalability are limited compared to machine learning
approaches on large-scale, intermittent retail demand.
